## Flight delays in TunisAir

This notebook is a response to the [flight delay prediction challenge in Zindi](https://zindi.world/competitions/flight-delay-prediction-challenge).
The challenge provides a training dataset including 10 variables. These contain nine predictors with information of the flights of the airline across a three-year span. The target variable is the flight delay (in minutes).

## Exploratory data analysis

We converted the date information into pythons datetime format and use it to create additional variables describing time ("hour", "weekday", "month", "year"). Spaces in variables aircraft (AC) and flight (FLTID) were removed.
The hour variable was re-classified into four time periods ("Morning", "Afternoon", "Evening", "Night") meaningful for airline operators. Similarly, the weekdays were reclassified into one delay-related variable "weekend" (True: Friday, Saturday, Sunday, Monday and False: Tuesday, Wednesday, Thursday).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
#from data_profiling import ProfileReport
import numpy as np
import datetime as dt

#Loading data
df = pd.read_csv('data/raw/Train.csv')

print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
df.info()


In [ ]:
# Quick summary of categorical variables
categorical_columns = ["FLTID","DEPSTN","ARRSTN","STATUS","AC"]

for column in categorical_columns:
    print(f"\n--- {column} ---")
    print("Unique values:", df[column].nunique())
    print(df[column].value_counts().head(5))

In [ ]:
# Finding leading/trailing whitespace in text columns
text_columns = df.select_dtypes(include=["object", "string"]).columns

for column in text_columns:
    original = df[column].astype(str)
    cleaned = original.str.strip()

    whitespace_rows = (original != cleaned).sum()

    print(f"{column}: {whitespace_rows} rows with leading/trailing spaces")

#print("Unique FLTID before cleaning:", df["FLTID"].nunique())

In [ ]:
df_clean = df.copy()

# Removing leading/trailing whitespace from text columns
df_clean["FLTID"] = df_clean["FLTID"].str.strip()
print("Unique FLTID after cleaning:", df_clean["FLTID"].nunique())

In [ ]:
df_clean["FLTID"] = df["FLTID"].str.strip()
df["FLTID"] = df["FLTID"].replace(" ", "", regex=True)
df["AC"] = df["AC"].replace(" ", "", regex=True)

In [ ]:
df_clean["DATOP"] = pd.to_datetime(df_clean["DATOP"],format="%Y-%m-%d")
#df_clean["DATOP"].head()
#df_clean["DATOP"].dtype

df_clean["STD"] = pd.to_datetime(df_clean["STD"],format="%Y-%m-%d %H:%M:%S") # STD using : as separator
df_clean["STD"].dtype

#STA using . as separator, this line will convert it to datetime format same as STD and DATOP
df_clean["STA"] = pd.to_datetime(df_clean["STA"], format="%Y-%m-%d %H.%M.%S") 
df_clean["STA"].dtype

## Importing Airport data

In [ ]:
import airportsdata
from zoneinfo import ZoneInfo
import datetime as dt

# Load airport data by IATA code
airports = airportsdata.load("IATA")

# Add timezone based on departure and arrival airport
df_clean["DEP_timezone"] = df_clean["DEPSTN"].map(
    lambda code: airports.get(code, {}).get("tz"))

df_clean["ARR_timezone"] = df_clean["ARRSTN"].map(
    lambda code: airports.get(code, {}).get("tz"))

#print(df_clean["DEP_timezone"].isna().sum())
#print(df_clean["ARR_timezone"].isna().sum())

print(df_clean.loc[df_clean["DEP_timezone"].isna(),"DEPSTN"].unique())
print(df_clean.loc[df_clean["ARR_timezone"].isna(),"ARRSTN"].unique())

# Historical airport (SXF) not found in current airportsdata
df_clean.loc[df_clean["DEPSTN"] == "SXF", "DEP_timezone"] = "Europe/Berlin"
df_clean.loc[df_clean["ARRSTN"] == "SXF", "ARR_timezone"] = "Europe/Berlin"

df_clean.columns

In [ ]:
print(df_clean.loc[df_clean["DEP_timezone"].isna(),"DEPSTN"].unique())
print(df_clean.loc[df_clean["ARR_timezone"].isna(),"ARRSTN"].unique())
df_clean.info()


### Converting to UTC

In [ ]:
def to_utc(dt, tz):
    if pd.isna(dt) or pd.isna(tz):
        return pd.NaT
    
    return (
        dt.replace(tzinfo=ZoneInfo(tz))
          .astimezone(ZoneInfo("UTC"))
    )


df_clean["STD_UTC"] = [
    to_utc(dt, tz)
    for dt, tz in zip(
        df_clean["STD"],
        df_clean["DEP_timezone"]
    )
]

df_clean["STA_UTC"] = [
    to_utc(dt, tz)
    for dt, tz in zip(
        df_clean["STA"],
        df_clean["ARR_timezone"]
    )
]



### Calculate flight duration in minutes

In [ ]:
df_clean["duration"] = (df_clean["STA_UTC"] - df_clean["STD_UTC"]).dt.total_seconds() / 60

print(df_clean["duration"].isna().sum())

### Weekday and season features

In [ ]:
df_clean["weekday"] = df_clean["STD_UTC"].dt.dayofweek
df_clean["weekend"] = df_clean["weekday"].map({0:True, 1:False, 2:False, 3:False, 4:True, 5:True, 6:True})
df_clean["season"] = df_clean["STD_UTC"].dt.month.map({
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn"
})


In [ ]:
df_clean['hour'] = df_clean["STD_UTC"].dt.hour
df_clean['weekday'] = df_clean["STD_UTC"].dt.weekday
df_clean['month'] = df_clean["STD_UTC"].dt.month
df_clean['year'] = df_clean["STD_UTC"].dt.year

 ### Adding the route

In [ ]:
df_clean["route"] = (df_clean["DEPSTN"] + "_" + df_clean["ARRSTN"])

Adding Rotation sequence

In [ ]:
df_clean = df_clean.sort_values(["DATOP","AC"]).copy()

df_clean["rotation_sequence"] = (df_clean.groupby(["DATOP", "AC"]).cumcount() + 1)

df_clean.head(10)

We used the estimated departure and arrival times to calculate the estimated duration of flights ("duration"). 

In [ ]:
# Interval between previous flight and next flight
df_clean = (
    df_clean
    .sort_values(["AC", "STD_UTC"])
    .copy()
)

df_clean["previous_STA_UTC"] = (
    df_clean
    .groupby("AC")["STA_UTC"]
    .shift(1)
)
df_clean["interval"] = (
    df_clean["STD_UTC"] - df_clean["previous_STA_UTC"]
).dt.total_seconds() / 60

## Additional features
We used available information to create a new variable describing the size of the destination airports as, due to operational bottlenecks, the size of an airport might be related to flight delays.

In [ ]:
# downloaded Airport size data from https://datacatalog.worldbank.org/search/dataset/0038117/global-airports
airports = pd.read_csv("data/airport_volume_airport_locations.csv")
airports = airports[["Orig", "TotalSeats"]]

# Three airports are missing from this database: 'BYJ', 'GAF', 'EBM'
# These are small airports, so we will fill them with the minimum of the other airports
missing_airports = pd.DataFrame({
    "Orig": ["BYJ", "GAF", "EBM"],
    "TotalSeats": [airports["TotalSeats"].quantile(0.25), 
                   airports["TotalSeats"].quantile(0.25), 
                   airports["TotalSeats"].quantile(0.25)]})
airports = pd.concat([airports, missing_airports], ignore_index=True)

df_clean = pd.merge(df_clean, airports, how="left", left_on="DEPSTN", right_on="Orig")
df_clean.rename(columns={'TotalSeats': 'DEPSIZE'}, inplace=True)
df_clean.drop(columns=["Orig"], inplace=True)



In [ ]:
df_clean = pd.merge(df_clean, airports, how="left", left_on="ARRSTN", right_on="Orig", suffixes=('_y', '_x'))
df_clean.rename(columns={'TotalSeats': 'ARRSIZE'}, inplace=True)
df_clean.drop(columns=["Orig"], inplace=True)
df_clean.head(5)

## Saving clean version (Ready to train)

In [ ]:
df_clean.to_csv("data/processed/Train_cleaned.csv", index=False)